In [26]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.cox import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis

import jinja2

import time
set_config(display="text")  # displays text representation of estimators

In [27]:
dataset = pd.read_csv("../data/data_final_noend.csv")

diagnosis_variables = []

with open("../data/diagnosis_final_noend.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final_noend.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [28]:
# Find duplicated patients with regional ln recurrence = 1
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [29]:
# Prepare data
data_x = survival_dataset[feature_columns]

data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
# Since every duplicated patient has a consistent event indicator (regional ln recurrence) I can use this method
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

In [30]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])

train_index, test_index = next(cv)

In [31]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [32]:
print("Train events:", survival_dataset.iloc[train_index]["Regional LN recurrence"].sum())
print("Val events:", survival_dataset.iloc[test_index]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0


In [33]:
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

# CV splitter for hyperparameter tuning
sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)

In [34]:
# Scale features for FINAL MODEL training and test evaluation
# NOTE: This scaling is ONLY for final model training after CV hyperparameter selection
# During CV, scaling happens INSIDE each fold (see cox_cross_validation function)
# Without scaling, L1 penalty affects features with different ranges unequally

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# convert scaled arrays back to DataFrame (preserve original columns and indices)
x_train_scaled = pd.DataFrame(x_train_scaled, columns=x_train.columns, index=x_train.index)
x_test_scaled = pd.DataFrame(x_test_scaled, columns=x_test.columns, index=x_test.index)

In [35]:
# Define hyperparameter ranges
alpha_range = np.logspace(-4, -1, 20)
l1_ratio_range = [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]

metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

# Run grid search
results, best_params, summary_df = cox_grid_search(
    X=x_train,  # Pass UNSCALED data
    y=y_train,
    cv_folds=sgkf_1,
    alpha_range=alpha_range,
    l1_ratio_range=l1_ratio_range,
    metrics=metrics,
    scaler_class='standard',  # or 'mixed' with binary/continuous features
    verbose=True
)


COX MODEL GRID SEARCH
L1 ratios: [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]
Alphas: 20 values from 1.00e-04 to 1.00e-01
Total combinations: 120
Scaler: standard

[1/120] l1=1.000, alpha=1.00e-04 (elapsed: 0.0min, est. remaining: 0.0min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


  ⚠️ Error: Input estimate contains NaN.
[2/120] l1=1.000, alpha=1.44e-04 (elapsed: 0.2min, est. remaining: 14.4min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


  ⚠️ Error: Input estimate contains NaN.
[3/120] l1=1.000, alpha=2.07e-04 (elapsed: 0.5min, est. remaining: 19.7min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


... (43 outputs left out)

c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:86: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)


  ⚠️ Error: Input estimate contains NaN.
[105/120] l1=0.001, alpha=4.28e-04 (elapsed: 7.5min, est. remaining: 1.1min)
  ⚠️ Error: Numerical error, because weights are too large. Consider increasing alpha.
[106/120] l1=0.001, alpha=6.16e-04 (elapsed: 7.5min, est. remaining: 1.0min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)


[107/120] l1=0.001, alpha=8.86e-04 (elapsed: 8.2min, est. remaining: 1.0min)
[108/120] l1=0.001, alpha=1.27e-03 (elapsed: 8.2min, est. remaining: 0.9min)
[109/120] l1=0.001, alpha=1.83e-03 (elapsed: 8.2min, est. remaining: 0.8min)
[110/120] l1=0.001, alpha=2.64e-03 (elapsed: 8.2min, est. remaining: 0.7min)
[111/120] l1=0.001, alpha=3.79e-03 (elapsed: 8.3min, est. remaining: 0.7min)
[112/120] l1=0.001, alpha=5.46e-03 (elapsed: 8.3min, est. remaining: 0.6min)
[113/120] l1=0.001, alpha=7.85e-03 (elapsed: 8.3min, est. remaining: 0.5min)
[114/120] l1=0.001, alpha=1.13e-02 (elapsed: 8.3min, est. remaining: 0.4min)
[115/120] l1=0.001, alpha=1.62e-02 (elapsed: 8.3min, est. remaining: 0.4min)
[116/120] l1=0.001, alpha=2.34e-02 (elapsed: 8.4min, est. remaining: 0.3min)
[117/120] l1=0.001, alpha=3.36e-02 (elapsed: 8.4min, est. remaining: 0.2min)
[118/120] l1=0.001, alpha=4.83e-02 (elapsed: 8.4min, est. remaining: 0.1min)
[119/120] l1=0.001, alpha=6.95e-02 (elapsed: 8.4min, est. remaining: 0.1min)

In [36]:
import pandas as pd
import numpy as np

# results = your_nested_dictionary
flat_data = []

for rank_ratio, alpha_dict in results.items():
    for alpha, df_stats in alpha_dict.items():
        # 1. Extract the Mean values
        means = df_stats.loc['Mean']
        
        # 2. Extract the Std values (optional, but good for thesis reporting)
        stds = df_stats.loc['Std']
        
        # 3. Combine into a single dictionary for this row
        row = {
            'rank_ratio': rank_ratio,
            'alpha': alpha,
            'c-index_mean': means['c-index'],
            'c-index-ipcw_mean': means['c-index-ipcw'],
            'int-brier-score_mean': means['int-brier-score'],
            'c-index_std': stds['c-index'],
            'c-index-ipcw_std': stds['c-index-ipcw'],
            'int-brier-score_std': stds['int-brier-score']
        }
        flat_data.append(row)

# 4. Create the DataFrame
df_results = pd.DataFrame(flat_data)

# Sort by the most important metric (IPCW C-index) to find the best model
df_results = df_results.sort_values(by='c-index-ipcw_mean', ascending=False)

print(df_results.head())

    rank_ratio     alpha  c-index_mean  c-index-ipcw_mean  \
61       0.010  0.000886      0.759772           0.656339   
76       0.001  0.000886      0.759355           0.656162   
77       0.001  0.001274      0.766886           0.654024   
58       0.100  0.048329      0.816935           0.653654   
62       0.010  0.001274      0.766765           0.653629   

    int-brier-score_mean  c-index_std  c-index-ipcw_std  int-brier-score_std  
61              0.010481     0.111924          0.168448             0.002870  
76              0.010490     0.112672          0.168956             0.002878  
77              0.010188     0.109536          0.172033             0.002599  
58              0.009285     0.056817          0.080724             0.002380  
62              0.010170     0.109964          0.172327             0.002584  


In [37]:
df_results.to_csv("../results/normal/cox_grid_search_results_noend.csv", index=False)

# Train and Test Final Model with Best Hyperparameters from Subsampling

In [ ]:
# Load best hyperparameters from grid search results
best_results = pd.read_csv("../results/normal/cox_grid_search_results_noend.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  l1_ratio (rank_ratio): {best_params['rank_ratio']}")
print(f"  alpha: {best_params['alpha']}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  l1_ratio (rank_ratio): 0.001
  alpha: 0.001832981
  c-index-ipcw_mean (CV): 0.7199



In [24]:
# Train final model with best hyperparameters (NO SUBSAMPLING)
# Scale the full training set
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)

final_model = CoxnetSurvivalAnalysis(
    l1_ratio=best_params['rank_ratio'],
    alpha_min_ratio=best_params['alpha'],
    fit_baseline_model=True,
    n_alphas=100,
)

final_model.fit(x_train_scaled, y_train)
print("Model trained successfully on full training set (no subsampling)!")

Model trained successfully on full training set (no subsampling)!


In [25]:
# Test the model on test set
x_test_scaled = scaler.transform(x_test)

# Calculate c-index
c_index = final_model.score(x_test_scaled, y_test)

# Calculate c-index IPCW
tau = y_train["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train, y_test, final_model.predict(x_test_scaled), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test_scaled)

# Use robust time filtering
max_time = min(y_train['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train, y_test, preds_at_times, times)

print("\nTest Set Performance (No Subsampling):")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")


Test Set Performance (No Subsampling):
  C-index: 0.8332
  C-index IPCW: 0.7066
  Integrated Brier Score: 0.0078
